# Day 7 — Final Model Comparison

Bu notebook'ta Naive, ARIMA, Prophet ve tuned XGBoost modelleri aynı tarih
aralığı ve aynı time-series cross-validation yapısı kullanılarak
karşılaştırılacaktır.

Amaç, modeller arasında adil bir karşılaştırma yaparak ChatGPT trend serisi
için en başarılı modeli belirlemektir.

In [1]:
import sys

sys.path.append("..")

import pandas as pd

from sklearn.model_selection import TimeSeriesSplit

from src.forecasting import (
    evaluate_naive_cv,
    evaluate_arima_cv,
    evaluate_prophet_cv,
    evaluate_xgb_recursive_with_change,
)

/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


In [ ]:
# Veri yükleme, veri yükleme fonk: read_csv
path= "../data/processed/google_trends_ai_3y_clean.csv"

data = pd.read_csv(
    path,
    index_col="date",
    parse_dates= True,
)

chatgpt= data["chatgpt"]

print(chatgpt.shape)
print(chatgpt.index[0]) # Zaman serisi verilerinde indeks sütunu olarak tarih (DatetimeIndex) kullandığımız için, bu kod çalıştırıldığında çıktır olarak serinin başladığı en eski tarihi verir.
print(chatgpt.index[-1])


print(f"İlk Tarih: {chatgpt.index[0]} | Değer: {chatgpt.iloc[0]}")
print(f"Son Tarih: {chatgpt.index[-1]} | Değer: {chatgpt.iloc[-1]}")


(157,)
2023-07-30 00:00:00
2026-07-26 00:00:00
İlk Tarih: 2023-07-30 00:00:00 | Değer: 13
Son Tarih: 2026-07-26 00:00:00 | Değer: 66



## **neden 149 haftaya indireceğiz**?

XGBoost'ta `8 lag` kullanıyoruz. İlk satırı düşün:

```text
2023-07-30
```

Bu tarihi tahmin etmek için önceki **8 haftaya** ihtiyacımız var ama veri burada başladığı için yok.

Aynı problem ilk 8 satırda devam ediyor. Ancak 9. haftaya geldiğimizde:

```text
2023-09-24
```

artık arkasında tam 8 haftalık geçmiş var.

Dolayısıyla:

```text
157 - 8 = 149
```

XGBoost'un kullanılabilir zaman aralığı:

```text
2023-09-24 → 2026-07-26
149 hafta
```

Prophet, ARIMA ve Naive'ın avantajlı olmaması için onları da **2023-09-24'ten başlatacağız**.

Şimdi yeni cell'e sadece bunu yaz:



Buradaki yeni şey `iloc[N_LAGS:]`.

`N_LAGS = 8` olduğuna göre Python bunu şöyle okur:

```python
chatgpt.iloc[8:]
```

Yani:

> indeks pozisyonu 8'den başla, sonuna kadar al.

Python sıfırdan saydığı için:

```text
0  → 1. satır
1  → 2. satır
...
7  → 8. satır
8  → 9. satır  ← buradan başlıyoruz
```

Bu hücreden **`(149,)` ve başlangıç `2023-09-24`** bekliyoruz.

Bunu çalıştır; sonra aynı `TimeSeriesSplit` yapısını kuracağız.


In [10]:

N_LAGS = 8

aligned_chatgpt = chatgpt.iloc[N_LAGS:]

print(aligned_chatgpt.shape)
print(aligned_chatgpt.index[0])
print(aligned_chatgpt.index[-1])


(149,)
2023-09-24 00:00:00
2026-07-26 00:00:00


Biz her fold’da 4 haftalık gelecek tahmini yapıyorduk ve toplam 12 fold kullanıyorduk.

In [ ]:
# Time-series cross-validation yapısını oluştur

tscv_aligned = TimeSeriesSplit(
    n_splits=12,   # Kaç farklı fold olacak?
    test_size=4,  # Her fold'da kaç haftayı test edeceğiz?
)



In [13]:
print(type(tscv_aligned))
print(type(aligned_chatgpt))

<class 'sklearn.model_selection._split.TimeSeriesSplit'>
<class 'pandas.Series'>


### 1. Naive Modeli 




In [14]:
naive_model= evaluate_naive_cv(

    series= aligned_chatgpt, #veri serimiz
    splitter= tscv_aligned # cross-validation yapımız.

)

In [16]:
naive_model

,fold,MAE,RMSE
0,1,4.50,6.708204
1,2,4.75,4.924429
2,3,1.75,1.936492
3,4,8.75,8.760708
4,5,11.25,14.008926
5,6,1.50,1.870829
6,7,5.25,5.722762
7,8,6.00,6.244998
8,9,2.75,3.041381
9,10,1.25,1.658312


burdaki mae ve rmse her bir deneme için ayrı ayrı hesaplanmış biz ortalamasını bulucaz.

In [17]:
# Naive modelinin ortalama hata değerlerini hesapla

naive_mae= naive_model["MAE"].mean()

naive_rmse= naive_model["RMSE"].mean()

print("Naive MAE:",naive_mae)

print("Naive RMSE:",naive_rmse)

Naive MAE: 4.541666666666667
Naive RMSE: 5.241398362212329


#### Naive Modeli Sonuç

Naive sonuçları hiç değişmedi.

Önceden de:

MAE  = 4.541666...
RMSE = 5.241398...

Şimdi 149 haftaya hizalayınca da aynısı çıktı.

Bunun nedeni şu: TimeSeriesSplit(n_splits=12, test_size=4) yine serinin son 48 haftasını test ediyor. İlk 8 haftayı silmemiz test dönemlerini değiştirmedi; sadece ilk fold'ların eğitim geçmişini biraz kısalttı.

Naive model ise tahmin yaparken uzun geçmişi öğrenmiyor. Temelde:

“En son gördüğüm değer neyse sonraki değer de odur.”

diyor. Bu yüzden 2023'teki ilk 8 haftanın kaldırılması Naive'ın tahminlerini etkilemedi.

### ARIMA Modeli

In [ ]:
# ARIMA(1, 1, 1) modelini aynı hizalanmış veri üzerinde değerlendir

arima_model = evaluate_arima_cv(

    series= aligned_chatgpt,
    splitter= tscv_aligned,
    order= (1,1,1)

)

/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init

In [19]:
arima_model.head()

,fold,MAE,RMSE
0,1,5.810190,7.716679
1,2,3.328008,3.527958
2,3,1.569615,1.736242
3,4,8.941422,8.951346
4,5,11.856744,14.454413


In [20]:
arima_mae= arima_model["MAE"].mean()

arima_rmse= arima_model["RMSE"].mean()

print("ARIMA MAE:", arima_mae)
print("ARIMA RMSE:", arima_rmse)

ARIMA MAE: 4.413942245524211
ARIMA RMSE: 5.00016866722027


#### ARIMA Sonuç

sonuç çok az kötüleşmiş ama neredeyse aynı:

Önceki:
MAE  ≈ 4.409
RMSE ≈ 4.998

Hizalanmış:
MAE  ≈ 4.414
RMSE ≈ 5.000

Yani ilk 8 haftayı çıkarmak ARIMA’yı pek etkilememiş. Fark sadece yaklaşık 0.005 MAE. Bu da karşılaştırmayı hizaladığımızda ARIMA sonucunun oldukça stabil kaldığını gösteriyor.

Şimdi sırada Prophet var. Aynı şekilde sen doldur:

### Prophet Modeli

In [25]:
prophet_model = evaluate_prophet_cv(
    series= aligned_chatgpt,
    splitter= tscv_aligned,
    changepoint_prior_scale= 1.0,
)

12:13:59 - cmdstanpy - INFO - Chain [1] start processing
12:13:59 - cmdstanpy - INFO - Chain [1] done processing
12:13:59 - cmdstanpy - INFO - Chain [1] start processing
12:13:59 - cmdstanpy - INFO - Chain [1] done processing
12:13:59 - cmdstanpy - INFO - Chain [1] start processing
12:13:59 - cmdstanpy - INFO - Chain [1] done processing
12:13:59 - cmdstanpy - INFO - Chain [1] start processing
12:13:59 - cmdstanpy - INFO - Chain [1] done processing
12:13:59 - cmdstanpy - INFO - Chain [1] start processing
12:13:59 - cmdstanpy - INFO - Chain [1] done processing
12:13:59 - cmdstanpy - INFO - Chain [1] start processing
12:13:59 - cmdstanpy - INFO - Chain [1] done processing
12:13:59 - cmdstanpy - INFO - Chain [1] start processing
12:13:59 - cmdstanpy - INFO - Chain [1] done processing
12:13:59 - cmdstanpy - INFO - Chain [1] start processing
12:13:59 - cmdstanpy - INFO - Chain [1] done processing
12:13:59 - cmdstanpy - INFO - Chain [1] start processing
12:13:59 - cmdstanpy - INFO - Chain [1]

In [26]:
# Prophet modelinin ortalama hata değerlerini hesapla

prophet_mae = prophet_model["MAE"].mean()

prophet_rmse = prophet_model["RMSE"].mean()


print("Prophet MAE:", prophet_mae)
print("Prophet RMSE:", prophet_rmse)

Prophet MAE: 4.2114053290725275
Prophet RMSE: 4.745405820071035


#### Prohet Sonuç

burada belirgin bir değişim var:

Önceki Prophet:
MAE  ≈ 4.303
RMSE ≈ 4.826

Hizalanmış Prophet:
MAE  ≈ 4.211
RMSE ≈ 4.745

Yani Prophet, ilk 8 haftayı eğitim geçmişinden çıkardığımızda daha iyi sonuç verdi.

Naive hiç değişmemişti, ARIMA neredeyse hiç değişmemişti; ama Prophet biraz iyileşti. Bunun mantıklı bir nedeni var: Prophet bütün eğitim geçmişinden trend yapısını öğreniyor. İlk 8 haftayı çıkarmamız test haftalarını değiştirmese bile, her fold'da Prophet'in gördüğü eğitim verisini değiştiriyor. Bu yeni başlangıç onun trendi biraz daha iyi öğrenmesine yol açmış olabilir.

### XGBoost Modeli

XGBoost için sadece aligned_chatgpt yetmiyordu. Önce 8 lag ve change feature’larından oluşan bir X ve y hazırlamamız gerekiyordu.

In [27]:
lag_data_8 = pd.DataFrame(
    {
        "target": chatgpt,
        **{f"lag_{i}": chatgpt.shift(i) for i in range(1, 9)}, #otomatik olarak lag_1 → lag_8 sütunlarını oluşturuyor. Baştaki ** ise oluşan sözlüğün içindekileri ana sözlüğün içine açıyor.
    }
)


#### Ek Bilgi:

O baştaki çift yıldız (`**`), Python'da **Dictionary Unpacking (Sözlük Açma / Çıkarma)** operatörüdür.

En basit anlatımıyla: Bir sözlüğün (dictionary) içindeki tüm **anahtar-değer (key-value)** çiftlerini tek tek çıkarıp, bulunduğu yere **isimlendirilmiş parametreler (keyword arguments)** olarak dağıtmaya yarar.

---

### ⚙️ Neden Kullanılır? (Örnekle Anlayalım)

Diyelim ki elinde şöyle bir sözlük var:

```python
parametreler = {"a": 1, "b": 2}

```

Bu değerleri bir fonksiyona parametre olarak göndermek istediğinde normalde şöyle yazman gerekir:

```python
fonksiyon(a=1, b=2)

```

İşte `**parametreler` yazdığında Python senin yerine o sözlüğü açar ve otomatik olarak `a=1, b=2` şekline getirir.

---

### 🛠️ Senin Kodunda Ne İşe Yarıyor?

Senin kodun muhtemelen `pd.DataFrame(...)` veya `.assign(...)` gibi bir yapının içinde şu şekilde duruyordur:

```python
lag_df = pd.DataFrame(
    {
        "y": chatgpt,
        **{f"lag_{i}": chatgpt.shift(i) for i in range(1, N_LAGS + 1)}
    }
)

```

İşte buradaki `**` işareti:

1. `{f"lag_{i}": chatgpt.shift(i)...}` ile üretilen `{"lag_1": ..., "lag_2": ...}` sözlüğünün **parantezlerini kırar / içindekileri dışarı çıkarır**.
2. Çıkan bu `lag_1`, `lag_2` sütunlarını, en dıştaki ana sözlüğün içine **bağımsız elemanlar olarak enjekte eder**.

---

### 💡 `**` Olmasaydı Ne Olurdu?

Eğer o `**` işaretini koymazsan, Python iç içe bir yapı oluşturur:

* **`**` İLE (Doğru Kullanım):**
```python
{
    "y": chatgpt,
    "lag_1": chatgpt.shift(1),
    "lag_2": chatgpt.shift(2)
}
# Sonuç: Bütün sütunlar aynı seviyede, tertemiz bir DataFrame oluşur.

```


* **`**` OLMADAN (Yanlış Kullanım):**
```python
{
    "y": chatgpt,
    "sözlük_içinde_sözlük": {"lag_1": ..., "lag_2": ...} # HATA!
}
# Sonuç: Pandas ne yapacağını bilemez ve hata verir veya sütun içine sözlük gömer.

```



Özetle `**`, *"Benim elimde bir sözlük var, bunun ambalajını aç ve içindeki elemanları bulunduğun ana yapıya dahil et"* demektir.

In [29]:
lag_data_8.head(9)

,target,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6,lag_7,lag_8
date,,,,,,,,,
2023-07-30,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-08-06,14,13.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-08-13,14,14.0,13.0,NaN,NaN,NaN,NaN,NaN,NaN
2023-08-20,15,14.0,14.0,13.0,NaN,NaN,NaN,NaN,NaN
2023-08-27,16,15.0,14.0,14.0,13.0,NaN,NaN,NaN,NaN
2023-09-03,16,16.0,15.0,14.0,14.0,13.0,NaN,NaN,NaN
2023-09-10,18,16.0,16.0,15.0,14.0,14.0,13.0,NaN,NaN
2023-09-17,18,18.0,16.0,16.0,15.0,14.0,14.0,13.0,NaN
2023-09-24,20,18.0,18.0,16.0,16.0,15.0,14.0,14.0,13.0


In [31]:
# Lag oluşturulamayan ilk 8 satırı kaldır

lag_data_8 = lag_data_8.dropna()


# Son haftadaki değişimi göster

lag_data_8["change_1"] = (
    lag_data_8["lag_1"]
    - lag_data_8["lag_2"]
)


# Bir önceki haftadaki değişimi göster

lag_data_8["change_2"] = (
    lag_data_8["lag_2"]
    - lag_data_8["lag_3"]
)

sıradaki adım X ve y'yi ayırmak.

X = modelin baktığı bilgiler / özellikler.
y = modelin tahmin etmeye çalıştığı cevap.

Bizim örneğimizde model şunlara bakıyor:

lag_1
lag_2
...
lag_8
change_1
change_2

Bunların hepsi X.

Modelin tahmin etmek istediği şey ise o haftanın gerçek chatgpt trend değeri, yani:

target

Bu da y.

In [32]:
# Modelin kullanacağı feature sütunlarını seç

X_8_change = lag_data_8[
    [
        # lag_1'den lag_8'e kadar olan sütunlar
        *[f"lag_{i}" for i in range(1,9)],

        "change_1",
        "change_2",
    ]
]


# Tahmin etmeye çalıştığımız hedef sütun

y_8_change = lag_data_8["target"]

boyut ve type kontrolü

In [35]:
print(type(X_8_change))
print(type(y_8_change))

print("X shape:", X_8_change.shape)

print("y shape:", y_8_change.shape)

<class 'pandas.DataFrame'>
<class 'pandas.Series'>
X shape: (149, 10)
y shape: (149,)


In [37]:
# Tuned XGBoost modelini değerlendir

xgb_model = evaluate_xgb_recursive_with_change(
    X=X_8_change,
    y=y_8_change,
    splitter=tscv_aligned,
)

#### XGBoost Sonuç

In [38]:
# ortalama hata değerini çıkarma

xgb_mae = xgb_model["MAE"].mean()

xgb_rmse = xgb_model["RMSE"].mean()


print("XGBoost MAE:", xgb_mae)
print("XGBoost RMSE:", xgb_rmse)

XGBoost MAE: 4.355783621470134
XGBoost RMSE: 4.901813134739656


In [60]:
all_mae= [naive_mae, arima_mae, prophet_mae, xgb_mae, ensemble_mae]

all_rmse= [naive_rmse, arima_rmse, prophet_rmse, xgb_rmse, ensemble_rmse]

comparison= pd.DataFrame(
    {
        "MAE": all_mae,
        "RMSE" : all_rmse
    },
    index=[
        "Naive",
        "ARIMA",
        "Prophet",
        "XGBoost",
        "Ensemble"
    ],
)

comparison

,MAE,RMSE
Naive,4.541667,5.241398
ARIMA,4.413942,5.000169
Prophet,4.211405,4.745406
XGBoost,4.355784,4.901813
Ensemble,3.911402,4.519721


### GENEL ANALİZ

In [42]:
comparison.sort_values(by="MAE", ascending= True)

,MAE,RMSE
Prophet,4.211405,4.745406
XGBoost,4.355784,4.901813
ARIMA,4.413942,5.000169
Naive,4.541667,5.241398


Hizalanmış 12-fold ve 4 haftalık time-series cross-validation sonuçlarına göre
Prophet modeli hem MAE hem de RMSE açısından en iyi performansı göstermiştir.

Evet, bu tablo bize artık sadece “hangi model kazandı?” değil, **verinin davranışı hakkında da ipucu** veriyor.

En önemli çıkarım şu: ChatGPT Google Trends serisinde **Prophet’in modellediği trend değişimleri**, sadece son haftalara veya sabit doğrusal ilişkilere bakmaktan biraz daha faydalı görünüyor.

Sonuçlarımız:

```text
Prophet  → MAE 4.211 | RMSE 4.745
XGBoost  → MAE 4.356 | RMSE 4.902
ARIMA    → MAE 4.414 | RMSE 5.000
Naive    → MAE 4.542 | RMSE 5.241
```

### Prophet neden önde olabilir?

Bunu “kesin sebep budur” diyemeyiz; bunu kanıtlamak için fold’lara tek tek bakmamız gerekir. Ama sonuçlardan güçlü bir hipotez çıkarabiliriz.

Prophet'in güçlü tarafı şu: serinin **genel trendini ve trendin zaman içinde değiştiği noktaları** modelleyebiliyor. Bizde de `changepoint_prior_scale=1.0` iyi çıkmıştı; yani Prophet'e trend değişimlerine karşı daha esnek olma izni verdiğimizde performansı arttı.

Mesela seri şöyle davranıyorsa:

```text
30 → 33 → 37 → 42 → 50 → 60 → 70
                         ↑
                  büyüme hızı değişiyor
```

Prophet bunu:

> “Trend hâlâ var ama artık aynı hızda ilerlemiyor.”

şeklinde modelleyebiliyor.

**XGBoost** ise çok yakın ikinci. O daha çok:

```text
son 8 hafta neydi?
son değişimler neydi?
geçmişte buna benzeyen durumlarda ne oldu?
```

diye bakıyor. Bu işe yarıyor ama bizim 4 haftalık tahminimiz **recursive** olduğu için ilk tahminini sonraki tahminde tekrar input olarak kullanıyor. İlk haftada biraz hata yaparsa:

```text
gerçek geçmiş
    ↓
tahmin 1
    ↓
tahmin 2'nin lag'i
    ↓
tahmin 3'ün lag'i
```

şeklinde hata ilerleyebiliyor.

**ARIMA** da gayet iyi. Fakat trendi fark alma (`d=1`) ve geçmiş değer/hatalar arasındaki daha düzenli doğrusal ilişkiler üzerinden öğreniyor. ChatGPT trendinde ani rejim/trend değişimleri varsa Prophet biraz daha esnek kalmış olabilir.

**Naive** ise:

> “Gelecek hafta ≈ bu hafta.”

diyor.

Buna rağmen `4.54` MAE alması aslında önemli. Bu, seride **haftadan haftaya güçlü bir süreklilik/persistence** olduğunu düşündürüyor. Yani trend çoğu hafta bir anda 20-30 puan zıplamıyor.

---

### Peki `MAE ≈ 4.21` iyi mi?

Burada önemli bir düzeltme: **4.21 bir “başarı skoru” değil, hata miktarı.** Dolayısıyla düşük olması iyi.

Google Trends değerlerimiz `0–100` ölçeğinde olduğuna göre:

```text
MAE = 4.21
```

şunu söylüyor:

> 4 haftalık tahminlerde Prophet'in tahminleri gerçek Google Trends değerlerinden ortalama yaklaşık **4.2 trend puanı** uzak kalmış.

Örneğin:

```text
Gerçek: 70
Tahmin: 74
Hata:   4

Gerçek: 65
Tahmin: 62
Hata:   3
```

gibi.

Ama dikkat: **“%4.2 arama hacmi hatası” diyemeyiz.** Google Trends'in `0–100` değerleri normalize edilmiş göreli ilgi değerleri; gerçek arama sayıları değil.

### İyi olup olmadığını en güzel nasıl anlarız?

Tek başına `4.21`e bakmak yerine **baseline ile karşılaştırırız.**

Naive:

```text
4.542
```

Prophet:

```text
4.211
```

Yani Prophet'in MAE'si Naive'a göre yaklaşık **%7.3 daha düşük**.

Bu olumlu bir sonuç, ama devasa bir fark değil.

Ayrıca:

```text
Prophet vs XGBoost

4.211 vs 4.356
```

arasında sadece yaklaşık **0.14 puan** var. Yani Prophet birinci olsa da XGBoost'u “kötü model” diye elemek kesinlikle doğru olmaz. İkisi oldukça yakın.

Ben bu sonuçlardan şu yorumu yapardım:

> **ChatGPT trend serisi hem güçlü kısa dönem sürekliliği içeriyor hem de zaman içinde değişen bir genel trende sahip görünüyor. Prophet bu trend değişimlerini en başarılı şekilde yakalarken, lag tabanlı tuned XGBoost çok yakın ikinci performansı göstermiştir.**

Bir sonraki adımda bence **ortalama sonuçlardan bir seviye aşağı inip fold bazında bakalım**. Çünkü asıl ilginç soru şu: Prophet her fold'da mı XGBoost'u yeniyor, yoksa bazı dönemlerde Prophet, bazı dönemlerde XGBoost mu daha iyi? Orası bize modellerin **neden** böyle sıralandığını çok daha iyi gösterecek.


In [ ]:
# Her fold için modellerin MAE değerlerini yan yana getir

fold_comparison = pd.DataFrame(
    {
        "Fold": list(range(1,13)),
        "Naive": naive_model["MAE"],
        "ARIMA": arima_model["MAE"],
        "Prophet": prophet_model["MAE"],
        "XGBoost": xgb_model["MAE"],
    }
)

fold_comparison

,Fold,Naive,ARIMA,Prophet,XGBoost
0,1,4.50,5.810190,4.364882,8.995085
1,2,4.75,3.328008,3.537106,1.165621
2,3,1.75,1.569615,1.479534,1.324392
3,4,8.75,8.941422,8.964874,8.410517
4,5,11.25,11.856744,7.888197,11.560068
5,6,1.50,2.098657,3.778760,1.898319
6,7,5.25,5.246729,6.164338,4.969233
7,8,6.00,4.466254,7.508301,2.932961
8,9,2.75,2.189144,3.011604,1.025185
9,10,1.25,1.305575,1.300534,1.305233


In [49]:

# 4 model sütununa bak, satır bazında (axis=1) en küçük değere sahip sütun adını "Winner"a yaz:
fold_comparison["Winner"] = fold_comparison[["Naive", "ARIMA", "Prophet", "XGBoost"]].idxmin(axis=1)

fold_comparison


,Fold,Naive,ARIMA,Prophet,XGBoost,Winner
0,1,4.50,5.810190,4.364882,8.995085,Prophet
1,2,4.75,3.328008,3.537106,1.165621,XGBoost
2,3,1.75,1.569615,1.479534,1.324392,XGBoost
3,4,8.75,8.941422,8.964874,8.410517,XGBoost
4,5,11.25,11.856744,7.888197,11.560068,Prophet
5,6,1.50,2.098657,3.778760,1.898319,Naive
6,7,5.25,5.246729,6.164338,4.969233,XGBoost
7,8,6.00,4.466254,7.508301,2.932961,XGBoost
8,9,2.75,2.189144,3.011604,1.025185,XGBoost
9,10,1.25,1.305575,1.300534,1.305233,Naive


In [50]:
winner_counts= fold_comparison["Winner"].value_counts()

winner_counts

Winner
XGBoost    6
Prophet    3
Naive      3
Name: count, dtype: int64

In [51]:
# Her modelin tek bir fold'da yaptığı en büyük MAE hatasını bul

worst_mae = fold_comparison[
    ["Naive", "ARIMA", "Prophet", "XGBoost"]
].max()

worst_mae

Naive      11.250000
ARIMA      11.856744
Prophet     8.964874
XGBoost    11.560068
dtype: float64

1. Çıkarım:

Worst MAE
Naive    → 11.25
ARIMA    → 11.86
Prophet  → 8.96
XGBoost  → 11.56


Burada en dikkat çekici şey **Prophet’in en kötü fold’unda bile diğer modellerden daha düşük maksimum hata yapması**.

Yani şimdilik şu çıkarımı yapabiliriz:

> **XGBoost daha fazla fold kazanıyor ama performansı daha dalgalı. Prophet daha az fold kazansa da kötü dönemlerde daha kontrollü kalıyor. Bu yüzden ortalama MAE ve RMSE’de Prophet öne geçiyor.**

Özellikle XGBoost için:

```text
6 fold galibiyeti
ama worst MAE ≈ 11.56
```

Prophet için:

```text
3 fold galibiyeti
ama worst MAE ≈ 8.96
```

Bu **“stabilite”** açısından Prophet lehine bir işaret.


Şimdi bunu bir adım daha ölçelim: sadece en kötü hataya değil, fold MAE’lerinin ne kadar değişken olduğuna bakalım.

Bunun için standart sapmayı kullanabiliriz.

In [53]:
# Fold'lar arasındaki MAE değişkenliğini ölç

mae_std = fold_comparison[
    ["Naive", "ARIMA", "Prophet", "XGBoost"]
].std()

mae_std.sort_values()

Prophet    2.813790
Naive      3.136720
ARIMA      3.259428
XGBoost    3.543698
dtype: float64

2. Çıkarım:

Yani XGBoost’un fold performansı en fazla dalgalanan model, Prophet’in ise en az dalgalanan model.

Bunu şöyle okuyabiliriz:

XGBoost bazı dönemlerde çok iyi tahmin yapıyor, ama bazı dönemlerde ciddi şekilde sapabiliyor. Prophet ise her zaman birinci olmasa da dönemler arasında daha tutarlı performans gösteriyor.

Şimdi çok ilginç bir şey daha kontrol edelim: ortalama yerine medyan MAE.

Çünkü sen az önce “XGBoost çoğu zaman daha iyi ama birkaç büyük hata ortalamasını bozuyor” dedin. Eğer bu doğruysa, medyanda XGBoost’un daha iyi görünme ihtimali var.

In [55]:
# Her modelin tipik fold performansını medyan ile karşılaştır

median_mae = fold_comparison[
    ["Naive", "ARIMA", "Prophet", "XGBoost"]
].median()

median_mae.sort_values()

XGBoost    3.547376
Prophet    3.657933
ARIMA      3.897131
Naive      4.625000
dtype: float64

3. Büyük Çıkarım ve Analiz

Bunu mantığa oturtmanın anahtarı şu: **fold’ları tablo satırı olarak değil, 12 ayrı “gerçek hayat provası” olarak düşünmek.** Şimdi sıfırdan veri analisti gibi okuyalım.

## 1. Biz aslında neyi simüle ediyoruz?

Diyelim bir şirkette çalışıyorsun ve yöneticin sana şöyle diyor:

> “Bugün elimizdeki Google Trends verilerine bakıp önümüzdeki yaklaşık 1 ayı tahmin et.”

Gerçek geleceği bilmiyorsun. Modeli geçmiş verilerle eğitip önümüzdeki 4 haftayı tahmin ediyorsun.

Biz geçmişe giderek bunu **12 farklı zamanda simüle ettik**.

Örneğin Fold 1 kabaca şu senaryo:

```text
Geçmiş veriler ────────────────┐
                               │ model burada
                               ▼
██████████████████████████████ | ???? ???? ???? ????
          TRAIN                  4 haftalık TEST
```

Model o `????` değerlerini bilmiyor. Tahmin ediyor. Sonra biz zaten geçmişte olduğumuz için gerçek değerleri biliyoruz ve:

> “Model ne kadar yanılmış?”

diye hesaplıyoruz.

Sonra birkaç hafta ileri gidiyoruz:

```text
██████████████████████████████████ | ???? ???? ???? ????
                    Fold 2
```

Bunu 12 kez yapıyoruz.

Dolayısıyla senin şu tablon:

| Fold | Naive | ARIMA | Prophet | XGBoost |
| ---- | ----: | ----: | ------: | ------: |
| 1    |  4.50 |  5.81 |    4.36 |    9.00 |
| 2    |  4.75 |  3.33 |    3.54 |    1.17 |
| ...  |   ... |   ... |     ... |     ... |

aslında şunu söylüyor:

> “Geçmişte 12 farklı zamanda gerçekten geleceği bilmiyormuşuz gibi davransaydık modeller nasıl performans gösterirdi?”

Bu, veri analistinin model değerlendirmesindeki ana düşünce.

---

# 2. Fold 2'yi gerçek hayat gibi okuyalım

Fold 2:

```text
Naive      4.75
ARIMA      3.33
Prophet    3.54
XGBoost    1.17
```

Bunu şöyle yorumluyoruz:

> “Bu dört haftalık dönemde XGBoost Google Trends değerlerini tahmin etmekte diğer modellerden çok daha başarılı olmuş.”

`MAE = 1.17` ne demek?

O dört haftadaki tahminler gerçek değerlerden ortalama yaklaşık **1.17 Google Trends puanı** uzak.

Mesela tamamen örnek olarak:

```text
Gerçek     Tahmin

80         79
82         84
85         84
83         82
```

gibi oldukça yakın bir tahmin dizisi olabilir.

Bu nedenle Fold 2 için:

> XGBoost bu dönemin yapısını çok iyi yakalamış.

diyoruz.

Ama burada henüz:

> “XGBoost en iyi model.”

diyemeyiz.

Çünkü önümüzde **11 başka gerçek hayat provası daha var.**

---

# 3. Fold 1'e bakınca neden fikrimiz değişiyor?

Fold 1:

```text
Naive      4.50
ARIMA      5.81
Prophet    4.36
XGBoost    9.00
```

Burada XGBoost tam tersine çok kötü.

Gerçek hayatta bu şu demek:

> “Eğer şirket tahmin sistemini tam bu dönemde XGBoost'a emanet etmiş olsaydı, tahminler ortalama yaklaşık 9 trend puanı sapacaktı.”

Prophet ise yaklaşık 4.36 sapmış.

Yani XGBoost:

```text
Fold 2 → müthiş
Fold 3 → çok iyi
...
Fold 1 → kötü
Fold 5 → kötü
```

davranıyor.

İşte veri analisti olarak burada sadece “kaç kez kazandı?”ya bakmayı bırakıyoruz.

---

# 4. “6 fold kazandı” aslında hangi sorunun cevabı?

XGBoost:

```text
6 / 12 fold kazandı
```

Bu şu soruya cevap veriyor:

> **“Hangi model en sık birinci oldu?”**

Cevap:

**XGBoost.**

Bu küçümsenecek bir bilgi değil. Hatta bize şunu düşündürüyor:

> XGBoost birçok normal dönemde verinin kısa vadeli yapısını oldukça iyi yakalıyor.

Medyan sonucumuz da bunu destekliyor:

```text
XGBoost median MAE  = 3.547
Prophet median MAE  = 3.658
```

Yani “tipik” bir dönem seçtiğimizde XGBoost hafifçe daha iyi.

Ama bu sadece bir boyut.

---

# 5. Ortalama MAE bize farklı bir soru soruyor

Ortalama:

```text
Prophet  = 4.211
XGBoost  = 4.356
```

Ortalamanın sorduğu soru kabaca şu:

> “Bu 12 dönemden rastgele birinde tahmin yapmak zorunda olsaydım, uzun vadede hangi modelden daha az hata beklerdim?”

Burada cevap:

**Prophet.**

Şimdi senin az önce fark ettiğin olay devreye giriyor.

XGBoost'un kazançları bazen şöyle:

```text
XGBoost   1.32
Prophet   1.48

Fark = 0.16
```

XGBoost kazanmış. Ama çok az.

Başka bir yerde:

```text
XGBoost   1.03
Prophet   3.01

Fark ≈ 1.98
```

Burada güzel bir kazanç var.

Ama kaybettiğinde:

```text
Fold 1

XGBoost   9.00
Prophet   4.36

Fark ≈ 4.64
```

Çok sert kaybediyor.

Dolayısıyla mantık:

```text
XGBoost

+0.2 avantaj
+0.5 avantaj
+1 avantaj
+2 avantaj
...

AMA

-4.6 dezavantaj
-3.7 dezavantaj
```

Birkaç büyük kötü sonuç küçük kazançları silebiliyor.

**İşte ortalama MAE bunu görüyor.**

---

# 6. Medyan neden XGBoost diyor?

Medyan:

```text
XGBoost   3.547
Prophet   3.658
```

Medyan uç değerlere fazla aldırmaz.

Şöyle hayali bir örnek düşün:

```text
Model A hataları:

2
2
3
3
3
4
4
4
5
5
10
15
```

Medyan yaklaşık:

```text
3.5
```

Ama ortalama son iki büyük hata yüzünden yükselir.

Medyan sana:

> “Bu modelin **tipik** performansı nasıl?”

sorusunu cevaplar.

O yüzden bizim sonuç:

> **Tipik bir dönemde XGBoost çok az farkla daha başarılı.**

demeye izin veriyor.

Ama:

> “Genel olarak daha güvenilir.”

demeye yetmiyor.

---

# 7. Standart sapma burada neden çok değerli?

Sonuç:

```text
Prophet  std = 2.814
XGBoost  std = 3.544
```

Standart sapmayı burada matematik formülü olarak değil, şöyle düşün:

> **“Modelin performansı dönemden döneme ne kadar sallanıyor?”**

Düşük:

```text
3 → 4 → 3 → 5 → 4 → 3
```

Daha stabil.

Yüksek:

```text
1 → 9 → 2 → 11 → 1 → 4
```

Daha oynak.

Dolayısıyla:

```text
XGBoost std = 3.54
```

bize:

> “XGBoost'un ne kadar iyi çalışacağını bulunduğumuz döneme bağlı olarak tahmin etmek daha zor.”

diyor.

Prophet:

```text
std = 2.81
```

olduğu için daha az oynak.

Gerçek şirket açısından bu çok önemli.

---

# 8. Worst MAE'nin gerçek hayattaki anlamı

Biz:

```text
Prophet worst  = 8.96
XGBoost worst  = 11.56
ARIMA worst    = 11.86
Naive worst    = 11.25
```

bulduk.

Bu metrik şu soruya cevap veriyor:

> **“İşler kötü giderse ne kadar kötü gidebilir?”**

Bunu risk yönetimi gibi düşün.

Mesela tahmin motorunu stok planlamasında kullandığını düşün.

Model A:

```text
genellikle çok iyi
ama bazen aşırı yanlış
```

Model B:

```text
genellikle biraz daha az mükemmel
ama felaket hata daha az
```

İşletme şunu söyleyebilir:

> “Benim için birkaç puan ekstra doğruluk değil, büyük yanlış tahmin yapmamak daha önemli.”

O zaman **Model B** daha değerlidir.

Bizde şu anda Prophet biraz bu karakteri gösteriyor.

---

# 9. RMSE neden aynı hikâyeyi destekliyor?

Hatırlarsan:

```text
Prophet RMSE = 4.745
XGBoost RMSE = 4.902
```

RMSE'nin önemli özelliği:

**Büyük hataları MAE'den daha fazla cezalandırması.**

Örneğin iki hata:

```text
2 ve 2
```

ile:

```text
0 ve 4
```

MAE açısından benzer olabilir.

Ama RMSE ikinci durumda `4`lük büyük hatayı daha fazla cezalandırır.

Dolayısıyla XGBoost'un bazı dönemlerde sert sapması varsa RMSE de bundan rahatsız olur.

Ve gerçekten:

```text
Prophet RMSE < XGBoost RMSE
```

bulduk.

Bu da bizim:

> “XGBoost'un tail/büyük hata problemi biraz daha fazla.”

yorumumuzla uyumlu.

---

# 10. Peki bu verinin kendisi hakkında ne öğreniyoruz?

İşte artık gerçekten **veri analisti kısmı** geliyor.

Naive modeli bile:

```text
MAE = 4.54
```

aldı.

Naive'ın zekâsı yok denecek kadar az:

> “Son gördüğüm değer neyse gelecek de yaklaşık odur.”

Buna rağmen Prophet:

```text
4.21
```

XGBoost:

```text
4.36
```

Naive:

```text
4.54
```

Yani inanılmaz büyük farklar yok.

Bu bize ChatGPT Google Trends serisi hakkında önemli bir ipucu veriyor:

> **Seride güçlü kısa dönem sürekliliği var.**

Başka bir ifadeyle:

```text
Bu hafta = 70
```

ise gelecek birkaç haftanın tamamen:

```text
20
95
15
100
```

gibi kaotik davranma ihtimali düşük.

Yakın geçmiş gelecek için zaten güçlü bir bilgi taşıyor.

Bu nedenle çok basit Naive bile şaşırtıcı derecede iyi.

Bu, bir veri analistinin rapora yazabileceği gerçek bir bulgudur.

---

# 11. Prophet'in önde olması bize veri hakkında ne düşündürüyor?

Burada dikkatli dil kullanıyoruz.

**“Prophet kazandı, demek ki kesin şu özellik var” diyemeyiz.**

Ama hipotez kurabiliriz.

Prophet'in güçlü olduğu şeylerden biri trendin zaman içinde değişmesini modellemek.

Biz ayrıca Prophet tuning sırasında:

```text
changepoint_prior_scale = 1.0
```

ile daha esnek trend modelinin daha iyi çalıştığını gördük.

Bu iki bulgu birlikte bize şunu düşündürüyor:

> ChatGPT arama ilgisi yalnızca “son birkaç hafta ne oldu?” bilgisine bağlı değil; daha geniş zaman ölçeğindeki trend değişimleri de tahmin için yararlı olabilir.

Bu **kanıtlanmış fizik yasası değil**, veri analisti hipotezi.

İyi analist dili böyle olur:

❌

> “Prophet iyi olduğu için veride kesin trend kırılması vardır.”

✅

> “Prophet'in esnek trend yapısıyla daha iyi performans göstermesi, seride zaman içinde değişen trend yapısının tahmin açısından önemli olabileceğini düşündürmektedir.”

Bu ayrım çok önemli.

---

# 12. XGBoost bize ne anlatıyor?

XGBoost'un:

```text
6 fold kazanması
median MAE'de 1. olması
```

başka bir ipucu veriyor.

XGBoost'a verdiğimiz şeyler:

```text
lag_1 ... lag_8
change_1
change_2
```

Yani esas olarak **son 8 haftalık davranış**.

Buna rağmen çok başarılı.

Bu da şunu düşündürüyor:

> Yakın geçmişteki değerler ve kısa vadeli değişim örüntüleri gelecek 4 hafta için ciddi tahmin gücü taşıyor.

Yani veride hem:

```text
uzun dönem trend bilgisi
```

hem de:

```text
kısa dönem yerel pattern
```

önemli olabilir.

Prophet birincisine daha doğal yaklaşırken XGBoost ikincisini çok iyi kullanıyor.

---

# 13. ARIMA'nın hiç fold kazanmaması “kötü model” anlamına geliyor mu?

Hayır.

ARIMA:

```text
MAE = 4.414
```

XGBoost:

```text
4.356
```

Aralarında sadece:

```text
≈ 0.058
```

var.

Yani ARIMA hiçbir fold'da tam birinci olmayabilir ama **genel olarak rekabetçi**.

Bu biraz yarış gibi:

```text
Koşucu A:
3., 2., 3., 2., 3., 2...

Koşucu B:
1., 7., 1., 8., 1...
```

Koşucu A hiç yarış kazanmamış olabilir ama genel performansı çok sağlam olabilir.

Dolayısıyla:

> “ARIMA sıfır fold kazandı, işe yaramıyor.”

yanlış çıkarım olur.

---

# 14. “MAE 4.21 iyi mi?” sorusuna analist gibi cevap

Tek başına:

```text
4.21
```

görüp “iyi” diyemeyiz.

Çünkü ölçeği bilmemiz gerekir.

Bizim Google Trends:

```text
0–100
```

normalize ölçeğinde.

Dolayısıyla yaklaşık:

> Ortalama 4.2 **trend puanı** hata.

Ama esas değerlendirme baseline'a karşı.

Naive:

```text
4.542
```

Prophet:

```text
4.211
```

Prophet'in MAE iyileşmesi yaklaşık:

```text
(4.542 - 4.211) / 4.542
≈ %7.3
```

Yani Prophet basit Naive baseline'a göre ortalama hatayı yaklaşık **%7 azaltıyor**.

RMSE açısından iyileşme de yaklaşık **%9.5**.

Bu:

> “Model gerçekten baseline'dan daha iyi.”

dememize destek veriyor.

Ama:

> “Muazzam devrimsel iyileşme.”

değil.

Ve önemli: bu **%7 daha fazla gerçek arama tahmini doğruluğu** demek değildir. Google Trends puanları normalize.

---

# 15. Şu an bir şirkette olsaydın hangi modeli seçerdin?

Şu anda elimizdeki verilere dayanarak ben rapora şöyle yazardım:

> **Prophet ana model adayıdır.**

Ama neden?

Sadece “MAE en düşük” diye değil.

Elimizdeki bütün kanıt:

| Özellik             | Daha iyi |
| ------------------- | -------- |
| Ortalama MAE        | Prophet  |
| Ortalama RMSE       | Prophet  |
| Medyan MAE          | XGBoost  |
| Fold kazanma sayısı | XGBoost  |
| Fold değişkenliği   | Prophet  |
| En kötü fold        | Prophet  |

Yani karar şuna dönüşüyor:

**XGBoost:** tipik dönemlerde çok güçlü, daha sık kazanan, fakat daha riskli.

**Prophet:** biraz daha tutarlı, büyük hataları daha iyi sınırlayan ve bütün dönemler hesaba katıldığında en düşük ortalama hatayı veren model.

Eğer tahmin motorunun amacı:

> “Her ay makul derecede güvenilir tahmin üret.”

ise Prophet şu anda daha doğal seçim.

Ama amaç:

> “Çoğu dönemde mümkün olan en isabetli tahmini istiyorum ve ara sıra büyük hata riskini kabul ederim.”

ise XGBoost ciddi aday.

İşte **model seçimi sadece teknik değil, iş problemine bağlı** dediğimiz şey tam olarak bu.

---

## 16. Fakat iyi bir veri analisti burada “Prophet kesin kazandı” demez

Çünkü bizim hâlâ iki metodolojik sınırlamamız var.

Birincisi, sadece **12 fold** görüyoruz. Çok büyük bir örneklem değil.

İkincisi, bazı hyperparameter'ları da bu CV sonuçlarına bakarak seçtik. Dolayısıyla aynı fold'ları hem model ayarlamak hem performans değerlendirmek için kullandığımız ölçüde sonuçlarımız biraz iyimser olabilir.

Bu yüzden şu an doğru ifade:

> **“Mevcut cross-validation sonuçlarında Prophet en güçlü genel aday görünmektedir.”**

“Prophet kesin en iyi modeldir” değil.

Bu veri analisti dilindeki en önemli alışkanlıklardan biridir: **çıktının söylediğinden fazlasını iddia etmemek.**

---



### Ensemble Model Denemesi

Mevcut değerlendirmede `n_splits=12` ve `test_size=4` kullanılarak son 48 haftayı
kapsayan 12 farklı 4 haftalık tahmin dönemi test edilmiştir. Bu yapı yaklaşık
bir yıllık farklı dönemlerde model performansını gözlemlemeye izin verdiği için
şimdilik `n_splits` değeri artırılmayacaktır. Daha fazla split kullanmak yeni
veri oluşturmayacağı gibi, ilk fold'larda kullanılan eğitim verisini de
azaltacaktır.

Fold bazlı sonuçlar Prophet ve XGBoost modellerinin farklı dönemlerde farklı
davranışlar gösterdiğini ortaya koymuştur.

- Prophet daha düşük ortalama MAE ve RMSE değerlerine sahiptir.
- Prophet'in fold'lar arasındaki hata değişkenliği daha düşüktür.
- XGBoost daha fazla fold'da en iyi model olmuştur.
- XGBoost'un medyan MAE değeri Prophet'ten biraz daha düşüktür.
- Ancak XGBoost bazı dönemlerde daha büyük hatalar yapmaktadır.

Bu sonuçlar iki modelin birbirini tamamlayıp tamamlayamayacağını araştırmayı
anlamlı hale getirmektedir.

Bu nedenle bir sonraki adımda Prophet ve XGBoost tahminlerinin eşit ağırlıklı
ortalaması alınarak basit bir ensemble model oluşturulacaktır:

**Ensemble Prediction = 0.5 × Prophet Prediction + 0.5 × XGBoost Prediction**

Ensemble modeli de aynı 12-fold ve 4 haftalık cross-validation yapısı üzerinde
değerlendirilecektir. Amaç, iki modelin farklı dönemlerde yaptığı hataların
birbirini dengeleyip dengelemediğini gözlemlemektir.

Ensemble modelinin MAE ve RMSE değerleri Prophet ve XGBoost modellerinden daha
düşük çıkarsa iki modelin birlikte kullanılmasının faydalı olduğu sonucuna
varılacaktır. Aksi durumda daha karmaşık bir model oluşturmadan en başarılı tek
model kullanılacaktır.

In [56]:
def evaluate_ensemble_cv(
    series,
    X,
    y,
    splitter,
    n_lags=8,
    prophet_weight=0.5,
    xgb_weight=0.5,
):
    """
    Evaluate a Prophet + XGBoost ensemble using
    time-series cross-validation.
    """

    results = []

    for fold, (train_index, test_index) in enumerate(
        splitter.split(X),
        start=1,
    ):

        # Bu fold'daki gerçek test değerlerini al
        y_test = y.iloc[test_index]

        # Bu fold'daki XGBoost eğitim verilerini al
        X_train = X.iloc[train_index]

        y_train = y.iloc[train_index]

In [ ]:
# forecasting.py dosyasının güncel halini notebook'a tekrar yükle

import importlib
import src.forecasting

importlib.reload(src.forecasting) #“Bu modülü daha önce yüklemiştim ama dosyayı değiştirdim; diskteki yeni halini tekrar oku.”


# Yeni ensemble fonksiyonunu import et

from src.forecasting import evaluate_ensemble_cv

In [58]:
# Prophet + XGBoost eşit ağırlıklı ensemble'ı değerlendir

ensemble_model = evaluate_ensemble_cv(
    series= aligned_chatgpt,
    X= X_8_change,
    y=y_8_change,
    splitter= tscv_aligned,
    prophet_weight= 0.5,
    xgb_weight= 0.5,
)

15:19:16 - cmdstanpy - INFO - Chain [1] start processing
15:19:16 - cmdstanpy - INFO - Chain [1] done processing
15:19:16 - cmdstanpy - INFO - Chain [1] start processing
15:19:16 - cmdstanpy - INFO - Chain [1] done processing
15:19:16 - cmdstanpy - INFO - Chain [1] start processing
15:19:16 - cmdstanpy - INFO - Chain [1] done processing
15:19:17 - cmdstanpy - INFO - Chain [1] start processing
15:19:17 - cmdstanpy - INFO - Chain [1] done processing
15:19:17 - cmdstanpy - INFO - Chain [1] start processing
15:19:17 - cmdstanpy - INFO - Chain [1] done processing
15:19:17 - cmdstanpy - INFO - Chain [1] start processing
15:19:17 - cmdstanpy - INFO - Chain [1] done processing
15:19:17 - cmdstanpy - INFO - Chain [1] start processing
15:19:17 - cmdstanpy - INFO - Chain [1] done processing
15:19:17 - cmdstanpy - INFO - Chain [1] start processing
15:19:17 - cmdstanpy - INFO - Chain [1] done processing
15:19:18 - cmdstanpy - INFO - Chain [1] start processing
15:19:18 - cmdstanpy - INFO - Chain [1]

#### Ensemble Model Sonuçları

In [59]:
# Ensemble modelinin ortalama hata değerlerini hesapla

ensemble_mae = ensemble_model["MAE"].mean()

ensemble_rmse = ensemble_model["RMSE"].mean()


print("Ensemble MAE:", ensemble_mae)
print("Ensemble RMSE:", ensemble_rmse)

Ensemble MAE: 3.911401565392208
Ensemble RMSE: 4.519721160387151


In [61]:
comparison["MAE"].sort_values()

Ensemble    3.911402
Prophet     4.211405
XGBoost     4.355784
ARIMA       4.413942
Naive       4.541667
Name: MAE, dtype: float64

#### Ensemble Model Çıkarım

Evet, ensemble gerçekten işe yaramış. Hem MAE hem RMSE’de tek başına Prophet ve XGBoost’u geçti:

Model       MAE      RMSE
Ensemble    3.911    4.520   ← en iyi
Prophet     4.211    4.745
XGBoost     4.356    4.902
ARIMA       4.414    5.000
Naive       4.542    5.241

Buradaki önemli çıkarım şu: Prophet ve XGBoost aynı haftalarda aynı miktarda hata yapmıyorlar. Biri bir haftayı fazla tahmin ederken diğeri daha düşük tahmin edebiliyor; %50-%50 ortalama alınca bu hatalar kısmen birbirini dengeliyor.

Örneğin:

Gerçek     = 70
Prophet    = 66
XGBoost    = 74

Ensemble   = 70

Tek tek modeller hata yapmasına rağmen birlikte doğruya yaklaşabiliyorlar.

Prophet’e göre ensemble’ın MAE’si yaklaşık 0.30 puan, yani %7 civarı daha düşük. XGBoost’a göre iyileşme ise yaklaşık %10. Bu artık küçük bir ondalık farktan daha anlamlı bir gelişme.

Ama şimdi veri analisti olarak hemen “Ensemble kesin en iyi model!” demiyoruz. Ortalama çok iyi çıktı ama 12 fold’un içinde ne yaptığını görmemiz lazım. Mesela ortalamayı birkaç muhteşem fold mu aşağı çekiyor, yoksa gerçekten daha stabil mi?

In [70]:
# Her fold için Ensemble MAE değerlerini mevcut karşılaştırmaya ekle

fold_comparison["Winner"] = fold_comparison[
    ["Naive", "ARIMA", "Prophet", "XGBoost", "Ensemble"]
].idxmin(axis=1)

fold_comparison


,Fold,Naive,ARIMA,Prophet,XGBoost,Ensemble,Winner
0,1,4.50,5.810190,4.364882,8.995085,7.953726,Prophet
1,2,4.75,3.328008,3.537106,1.165621,1.207810,XGBoost
2,3,1.75,1.569615,1.479534,1.324392,1.371816,XGBoost
3,4,8.75,8.941422,8.964874,8.410517,8.687695,XGBoost
4,5,11.25,11.856744,7.888197,11.560068,7.876231,Ensemble
5,6,1.50,2.098657,3.778760,1.898319,2.315463,Naive
6,7,5.25,5.246729,6.164338,4.969233,5.566785,XGBoost
7,8,6.00,4.466254,7.508301,2.932961,4.960974,XGBoost
8,9,2.75,2.189144,3.011604,1.025185,1.663433,XGBoost
9,10,1.25,1.305575,1.300534,1.305233,1.302883,Naive


In [71]:
# Ensemble'ın tipik, en kötü ve değişken performansını ölç

ensemble_median = fold_comparison["Ensemble"].median()

ensemble_worst = fold_comparison["Ensemble"].max()

ensemble_std = fold_comparison["Ensemble"].std()


print("Median:", ensemble_median)
print("Worst:", ensemble_worst)
print("Std:", ensemble_std)

Median: 2.3419122244200654
Worst: 8.687695431005121
Std: 2.9282510159936885


Evet, şimdi ensemble’ın davranışı daha net okunuyor. Sonuçlar:

```text
Ensemble
Ortalama MAE = 3.911
Medyan MAE   = 2.342
Worst MAE    = 8.688
Std          = 2.928
RMSE         = 4.520
```

Bunları tek tek veri analisti gibi yorumlayalım.

**Medyan = 2.34** özellikle çok iyi bir sonuç. Önceden medyanlar:

```text
XGBoost  = 3.547
Prophet  = 3.658
Ensemble = 2.342
```

Medyan bize “12 fold’un ortasındaki, daha tipik performans nasıl?” sorusunu cevaplıyor. Yani ensemble sadece birkaç şanslı fold yüzünden ortalamada iyi çıkmamış; **çoğu normal dönemde de hata seviyesi ciddi şekilde düşmüş.**

**Worst = 8.69** ise en kötü 4 haftalık dönemde bile MAE’nin yaklaşık 8.69 olduğunu söylüyor:

```text
Prophet  worst = 8.965
XGBoost  worst = 11.560
Ensemble worst = 8.688
```

Bu da güzel çünkü ensemble, XGBoost’un büyük sapmalarını bastırmış ve hatta Prophet’in en kötü sonucundan da biraz daha iyi kalmış.

**Std = 2.93** için:

```text
Prophet  = 2.814
Ensemble = 2.928
XGBoost  = 3.544
```

Burada Prophet hâlâ biraz daha stabil. Yani ensemble’ın fold’dan fold’a hata miktarı Prophet’ten çok az daha fazla oynuyor. Fakat XGBoost’tan belirgin biçimde daha stabil.

Dolayısıyla ensemble için şu resmi çiziyoruz:

> **Tipik dönemde en düşük hata, genel ortalamada en düşük hata, en kötü durumda da oldukça kontrollü hata.**

Bu, şu ana kadarki en güçlü model sonucu.

---

Şimdi çok güzel soruna gelelim:

## Fold 5’te ensemble nasıl Prophet’i bile geçebiliyor?

Fold 5:

```text
Prophet   MAE = 7.888197
XGBoost   MAE = 11.560068
Ensemble  MAE = 7.876231
```

İlk bakışta insan şöyle düşünüyor:

> “Ensemble Prophet ile XGBoost’un ortalamasıysa, nasıl Prophet’ten daha iyi olabilir?”

Çünkü biz **MAE değerlerini ortalamıyoruz**.

Şunu yapmıyoruz:

```text
(7.888 + 11.560) / 2
```

Biz **her haftanın tahminlerini önce ortalıyoruz**, ondan sonra gerçek değerle karşılaştırıyoruz.

Çok basit tek haftalık örnek:

```text
Gerçek değer = 80

Prophet tahmini = 75
XGBoost tahmini = 85
```

Hatalar:

```text
Prophet hata = |80 - 75| = 5
XGBoost hata = |80 - 85| = 5
```

Ama ensemble:

```text
(75 + 85) / 2 = 80
```

oluyor.

Ensemble hatası:

```text
|80 - 80| = 0
```

Yani:

```text
Prophet hata  = 5
XGBoost hata  = 5
Ensemble hata = 0
```

**Ensemble ikisini de geçti.**

Neden?

Çünkü modeller **gerçeğin iki farklı tarafında hata yaptı**:

```text
75 -------- 80 -------- 85
Prophet    Gerçek      XGBoost
```

İkisini ortalayınca doğrudan gerçeğe yaklaştık.

---

Fold 5’te de muhtemelen buna benzer bir şey bazı haftalarda oluyor.

Mesela tamamen örnek olsun:

```text
Gerçek    Prophet    XGBoost

80        74         87
85        77         91
...
```

XGBoost genel olarak Prophet’ten kötü olabilir. Ama bazı haftalarda Prophet **fazla düşük**, XGBoost **fazla yüksek** tahmin etmişse:

```text
Prophet -------- Gerçek -------- XGBoost
```

ortalama tahmin gerçeğe doğru çekilir.

Fold 5 ensemble’ın Prophet’i geçme farkı zaten çok küçük:

```text
Prophet  = 7.888
Ensemble = 7.876

fark ≈ 0.012
```

Yani XGBoost, Prophet’i mucizevi biçimde düzeltmemiş. Bazı haftalarda Prophet’in hatasını biraz azaltmış; başka haftalarda biraz bozmuş; toplamda **çok küçük bir net iyileşme** oluşmuş.

---

Burada başka çok ilginç bir nokta var. Ensemble sadece **1 fold kazanmış**:

```text
XGBoost  → 6
Naive    → 3
Prophet  → 2
Ensemble → 1
```

ama ortalama ve medyanda **ensemble birinci**.

Bu nasıl oluyor?

Çünkü “Winner” şu soruyu soruyor:

> “Bu fold’da mutlak en iyi kim?”

Ensemble ise birçok fold’da birinci olmak yerine sürekli **2. veya iyi bir konumda** kalıyor.

Mesela Fold 2:

```text
XGBoost   1.166  ← kazandı
Ensemble  1.208  ← neredeyse aynı
Prophet   3.537
```

Ensemble kazanamadı ama kötü de değildi.

Fold 3:

```text
XGBoost   1.324
Ensemble  1.372
Prophet   1.480
```

Yine ikinci ama kazanana inanılmaz yakın.

Fold 9:

```text
XGBoost   1.025
Ensemble  1.663
Prophet   3.012
```

Yine güçlü.

Yani ensemble’ın karakteri şöyle:

```text
Çok sık 1. değil
ama
çok sık iyi bölgede
+
çok kötü çöküşleri daha az
```

Bu yüzden 12 dönemin tamamını hesaba kattığımızda:

```text
Ensemble MAE = 3.911
```

ile herkesi geçiyor.

Bence şu anda model açısından en önemli bulgumuz şu:

> **Prophet ve XGBoost birbirinin kopyası gibi hata yapmıyor. İki modelin tahmin hataları bazı dönemlerde birbirini dengelediği için, eşit ağırlıklı ensemble tek modellerden daha düşük genel hata üretiyor.**

Bu, ensemble yapmanın gerçekten mantıklı olduğuna dair elimizdeki ilk güçlü kanıt.


### Ensemble'da Çeşitli Ağırlıklar Deneme

Şimdi bence **ensemble’ı biraz doğrulayıp sonra kilitlemeliyiz**. Çünkü `3.911 MAE` çok iyi çıktı ama tek bir `0.5 / 0.5` kombinasyonu denedik.

Fakat burada 20 farklı ağırlık deneyip en küçüğü seçmek istemiyorum; o zaman yine bu 12 fold’a fazla uyum sağlamaya başlarız. Sadece küçük bir **sensitivity check** yapalım:

```text
Prophet   XGBoost
0.4       0.6
0.5       0.5   ← mevcut
0.6       0.4
0.7       0.3
```

Özellikle Prophet tek başına daha stabil olduğu için ona biraz daha fazla ağırlık vermek işe yarıyor mu görmek istiyorum.

Sonra sonuçlar mesela:

```text
0.4 / 0.6 → 4.02
0.5 / 0.5 → 3.91
0.6 / 0.4 → 3.88
0.7 / 0.3 → 3.95
```

gibi çıkarsa `0.6 / 0.4` aday olur. Ama farklar `0.01–0.02` gibi minicikse **daha basit olan 50/50’de kalırız**.

Sonra ensemble tuning’i kapatacağız. Ondan sonra sıra:

```text
ChatGPT modeli tamamlandı
        ↓
Gemini
        ↓
aynı Naive / ARIMA / Prophet / XGBoost / Ensemble
        ↓
Claude
        ↓
aynı karşılaştırma
```

Böylece asıl önemli soruya cevap vereceğiz:

> “ChatGPT’te iyi çalışan ensemble gerçekten başka trend serilerinde de iyi mi, yoksa sadece ChatGPT verisine özel mi?”




In [72]:
# Prophet ve XGBoost için birkaç farklı ensemble ağırlığını karşılaştır

weight_results = []

for prophet_weight in [0.4, 0.5, 0.6, 0.7]:

    xgb_weight = 1 - prophet_weight

    result = evaluate_ensemble_cv(
        series=aligned_chatgpt,
        X=X_8_change,
        y=y_8_change,
        splitter=tscv_aligned,
        prophet_weight=prophet_weight,
        xgb_weight=xgb_weight,
    )

    weight_results.append(
        {
            "Prophet Weight": prophet_weight,
            "XGBoost Weight": xgb_weight,
            "MAE": result["MAE"].mean(),
            "RMSE": result["RMSE"].mean(),
        }
    )

15:50:21 - cmdstanpy - INFO - Chain [1] start processing
15:50:21 - cmdstanpy - INFO - Chain [1] done processing
15:50:21 - cmdstanpy - INFO - Chain [1] start processing
15:50:22 - cmdstanpy - INFO - Chain [1] done processing
15:50:22 - cmdstanpy - INFO - Chain [1] start processing
15:50:22 - cmdstanpy - INFO - Chain [1] done processing
15:50:22 - cmdstanpy - INFO - Chain [1] start processing
15:50:22 - cmdstanpy - INFO - Chain [1] done processing
15:50:22 - cmdstanpy - INFO - Chain [1] start processing
15:50:22 - cmdstanpy - INFO - Chain [1] done processing
15:50:22 - cmdstanpy - INFO - Chain [1] start processing
15:50:22 - cmdstanpy - INFO - Chain [1] done processing
15:50:22 - cmdstanpy - INFO - Chain [1] start processing
15:50:22 - cmdstanpy - INFO - Chain [1] done processing
15:50:23 - cmdstanpy - INFO - Chain [1] start processing
15:50:23 - cmdstanpy - INFO - Chain [1] done processing
15:50:23 - cmdstanpy - INFO - Chain [1] start processing
15:50:23 - cmdstanpy - INFO - Chain [1]

In [75]:
weight_comparison = pd.DataFrame(
    weight_results
)

weight_comparison.sort_values(by="MAE")





,Prophet Weight,XGBoost Weight,MAE,RMSE
1,0.5,0.5,3.911402,4.519721
0,0.4,0.6,3.918330,4.555241
2,0.6,0.4,3.930205,4.506815
3,0.7,0.3,3.967717,4.524903


In [76]:
weight_comparison.sort_values(by="RMSE")

,Prophet Weight,XGBoost Weight,MAE,RMSE
2,0.6,0.4,3.930205,4.506815
1,0.5,0.5,3.911402,4.519721
3,0.7,0.3,3.967717,4.524903
0,0.4,0.6,3.918330,4.555241


`0.6 / 0.4` kombinasyonu RMSE açısından çok az daha iyi sonuç vermiş olsa da,
`0.5 / 0.5` kombinasyonu en düşük MAE değerine sahiptir. Ağırlıklar arasındaki
performans farkları oldukça küçük olduğu için daha basit, dengeli ve
yorumlanabilir olan `0.5 / 0.5` ensemble yapısının kullanılması tercih
edilmiştir.

### Sonuç

ChatGPT Google Trends serisi için mevcut cross-validation sonuçlarında en iyi
genel performans eşit ağırlıklı Prophet + XGBoost ensemble modeli tarafından
elde edilmiştir.

Bu sonuç yalnızca mevcut ChatGPT serisi için geçerlidir. Bir sonraki aşamada
aynı modelleme ve değerlendirme yaklaşımı Gemini ve Claude trend serilerine
uygulanarak ensemble yaklaşımının farklı serilerde de benzer şekilde başarılı
olup olmadığı incelenecektir.